# CNC Guard — entrenamendua eta emaitzen azalpena

Entregako koadernoa, 2026-11-12. Aurretik lokalki entrenatutako RF ereduaren txostena eta balidazio-puntuazioak erabiltzen ditu. Beheko gelaxkak berriz exekutatu dira; ez da entrenamendu historikoa gaur egindakotzat aurkezten.

## Helburua eta datuen muga
Egungo matxura-egoera sailkatzen dugu. Ez dugu ETA, RUL edo etorkizuneko matxura iragartzen. Irakaslearen 8.000 errenkadako CSV baimendua soilik erabiltzen da. 5.600 errenkada entrenatzeko, 1.200 garapen-balidaziorako eta aurretik ikuskatutako beste 1.200 baztertuta: azken multzoa ez da ebaluatzen.


In [1]:
from pathlib import Path
import os, json, hashlib
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, recall_score, precision_score
root = Path.cwd()
if not (root / 'outputs/cg08/report.json').exists():
    root = root.parent
os.chdir(root)
report = json.loads(Path('outputs/cg08/report.json').read_text())
manifest = json.loads(Path('outputs/api/manifest.json').read_text())
raw = Path('data/ai4i2020_train.csv').read_bytes()
assert hashlib.sha256(raw).hexdigest() == manifest['training_source_sha256']
df = pd.read_csv('data/ai4i2020_train.csv')
print('Errenkadak:', len(df))
print('Partizioak:', {k: len(v) for k, v in report['splits'].items()})
print('Entrenamenduaren bertsioak:', report['versions'])


Errenkadak: 8000
Partizioak: {'train': 5600, 'validation': 1200, 'excluded_previously_inspected_test': 1200}
Entrenamenduaren bertsioak: {'python': '3.11.18', 'sklearn': '1.5.0', 'numpy': '1.26.4', 'pandas': '2.2.2', 'joblib': '1.5.3'}


## Aurreprozesamendua eta desoreka
Bost sentsoreak eta `machine_type` erabiltzen dira. IDak eta denbora-markak ez dira ezaugarri prediktiboak; `failure_*` etiketak ere kanpoan geratzen dira. Zenbakizko hutsuneak medianarekin betetzen dira eta makina mota one-hot bidez kodetzen da. Transformazioak entrenamendu-tolesaren barruan doitzen dira.

Dummy eta erregresio logistikoa RF hautagaiekin konparatzen dira. RF hautagaiek ezaugarri fisikoak, klase-pisuak eta zuhaitzen sakonera alderatzen dituzte. Lau toleseko OOF average precision erabiltzen da RF aukeratzeko; F2 atalasea garapen-balidazioan doitzen da. Horregatik emaitzak optimistak izan daitezke: ez dira kanpoko azken test independente baten emaitzak.


In [2]:
train = df.loc[report['splits']['train']]
print('Entrenamenduko klaseak:')
print(train.machine_failure.value_counts().sort_index().to_string())
print('Hutsuneak:')
print(train[manifest['input_features']].isna().sum().to_string())
print('Aukeratutako konfigurazioa:')
print(json.dumps(report['search'][report['winner']], indent=2))


Entrenamenduko klaseak:
machine_failure
0    5410
1     190
Hutsuneak:
air_temp_K        0
process_temp_K    0
rpm               0
torque_Nm         0
tool_wear_min     0
machine_type      0
Aukeratutako konfigurazioa:
{
  "kind": "rf",
  "physical": true,
  "class_weight": null,
  "max_depth": 12,
  "oof_selection_metrics": {
    "threshold": 0.3025667388167388,
    "precision": 0.8691099476439791,
    "recall": 0.8736842105263158,
    "f1": 0.8713910761154856,
    "f2": 0.8727655099894848,
    "roc_auc": 0.9672468138924021,
    "average_precision": 0.9089647747455067,
    "confusion_matrix": [
      [
        5385,
        25
      ],
      [
        24,
        166
      ]
    ]
  }
}


## Entrenamendua erreproduzitzea
`scripts/experiment_rf.py` fitxategiak pipeline osoa eta entrenamendu-algoritmoa ditu; `scripts/experiment_anomaly.py` fitxategiak Isolation Forest bereizia entrenatzen du. Berriro entrenatzeko, erabili blokeo-fitxategiko ingurunea eta irteera berriak:

```bash
uv sync --frozen --python 3.11
uv run --frozen python -m scripts.experiment_rf --output outputs/entrega_rf_berria
uv run --frozen python -m scripts.experiment_anomaly --output outputs/entrega_if_berria
uv run --frozen python -m scripts.package_model --rf-dir outputs/entrega_rf_berria --anomaly-dir outputs/entrega_if_berria --output outputs/entrega_api_berria
```

Entregatutako modeloa `outputs/api/rf.pkl` da; `models/cnc_failure_model.pkl` haren kopia berdina da. Zerbitzuak RF, IF eta manifestua batera behar ditu. Ez kargatu modeloa scikit-learn bertsio desberdin batekin.


In [3]:
for name, expected in manifest['files'].items():
    assert hashlib.sha256((Path('outputs/api') / name).read_bytes()).hexdigest() == expected
assert report['model_sha256'] == manifest['files']['rf.pkl']
print('Modeloen SHA-256 hashak: zuzenak')
scores = pd.read_csv('outputs/cg08/validation.csv', float_precision='round_trip')
assert scores.source_row_index.tolist() == report['splits']['validation']
assert scores.label.tolist() == df.loc[scores.source_row_index, 'machine_failure'].tolist()
cutoff = manifest['risk_threshold']
pred = (scores.rf_score >= cutoff).astype(int)
print('F2 atalasea:', cutoff)
print('Nahaste-matrizea [[TN, FP], [FN, TP]]:')
print(confusion_matrix(scores.label, pred))
print('Sailkapen-txostena:')
print(classification_report(scores.label, pred, target_names=['Matxurarik ez', 'Matxura'], digits=4, zero_division=0))
print('Recall:', recall_score(scores.label, pred))
print('Precision:', precision_score(scores.label, pred))


Modeloen SHA-256 hashak: zuzenak
F2 atalasea: 0.19082281465428477
Nahaste-matrizea [[TN, FP], [FN, TP]]:
[[1150    9]
 [   9   32]]
Sailkapen-txostena:
               precision    recall  f1-score   support

Matxurarik ez     0.9922    0.9922    0.9922      1159
      Matxura     0.7805    0.7805    0.7805        41

     accuracy                         0.9850      1200
    macro avg     0.8864    0.8864    0.8864      1200
 weighted avg     0.9850    0.9850    0.9850      1200

Recall: 0.7804878048780488
Precision: 0.7804878048780488


## Interpretazioa eta enpresaren erabakia
Recall-ek benetako matxuretatik zenbat detektatzen diren adierazten du; precision-ek alertetatik zenbat diren zuzenak. FN bat detektatu gabeko matxura da eta FP bat alferrikako ikuskapena eragin dezakeen alerta. Accuracy bakarrik erabiltzea desegokia da, matxurak gutxiengoa direlako.

F2-k recall gehiago lehenesten du. APIaren kolore-politikak 0,15 eta 0,70 mugak ditu; `risk_alert` erabakiak manifestuko F2 atalasea erabiltzen du. Ez dira erabaki bera. IFren bitxitasuna ez da matxura-probabilitatea eta ez du RF alerta ezeztatzen. Mantentze-erabakia pertsona batek berrikusten du. 1.200 €/orduko geldialdi-kostuak ez du berez frogatzen sistemak aurrezkia lortu duenik.

## Mugak
Balidazioa berrerabilia da, puntuazioak ez daude industria-arrisku gisa kalibratuta eta ez dago etorkizuneko matxuren ebaluaziorik. Modelo historikoa mantendu da; koaderno honetako berriz kalkulatutako txostenak ez du datu berrien gaineko ebaluazioa ordezkatzen.
